#### - IMPORTS

In [1]:
import os, json, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

 #### - CNN definition

In [2]:

class HWConvBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        pool_size: int = 2,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride,
            padding=padding, bias=True,
        )
        self.relu = nn.ReLU(inplace=False)
        self.pool = nn.MaxPool2d(kernel_size=pool_size, stride=pool_size)

    def forward(self, x: torch.Tensor, return_raw: bool = False):
        after_conv = self.conv(x)
        after_relu = self.relu(after_conv)
        after_pool = self.pool(after_relu)
        if return_raw:
            raw = {
                "pre_relu":  after_conv.detach().clone(),
                "pre_pool":  after_relu.detach().clone(),
                "post_pool": after_pool.detach().clone(),
            }
            return after_pool, raw
        return after_pool


class CNN_BNS_HW(nn.Module):
    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        conv1_cfg: tuple = (32, 3, 1, 1),
        conv2_cfg: tuple = (64, 3, 1, 1),
        fc1_units: int = 256,
        dropout_rate: float = 0.5,
        input_hw: tuple = (28, 28),
    ):
        super().__init__()
        c1_out, c1_k, c1_s, c1_p = conv1_cfg
        c2_out, c2_k, c2_s, c2_p = conv2_cfg

        self.conv_block1 = HWConvBlock(in_channels, c1_out, c1_k, c1_s, c1_p)
        self.conv_block2 = HWConvBlock(c1_out, c2_out, c2_k, c2_s, c2_p)

        h_out    = input_hw[0] // 4
        w_out    = input_hw[1] // 4
        flat_dim = c2_out * h_out * w_out

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_dim, fc1_units),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate),
            nn.Linear(fc1_units, num_classes),
        )

    def forward(self, x: torch.Tensor, return_features: bool = False):
        fmaps = {}
        if return_features:
            fmaps["input"] = x.detach().clone()

        if return_features:
            x, raw1 = self.conv_block1(x, return_raw=True)
            fmaps["conv1.pre_relu"]  = raw1["pre_relu"]
            fmaps["conv1.pre_pool"]  = raw1["pre_pool"]
            fmaps["conv1.post_pool"] = raw1["post_pool"]
        else:
            x = self.conv_block1(x)

        if return_features:
            x, raw2 = self.conv_block2(x, return_raw=True)
            fmaps["conv2.pre_relu"]  = raw2["pre_relu"]
            fmaps["conv2.pre_pool"]  = raw2["pre_pool"]
            fmaps["conv2.post_pool"] = raw2["post_pool"]
        else:
            x = self.conv_block2(x)

        if return_features:
            fmaps["flatten"] = x.detach().clone().view(x.size(0), -1)

        logits = self.classifier(x)
        if return_features:
            return logits, fmaps
        return logits

#### - Moduli utilities

In [3]:
def gcd(a: int, b: int) -> int:
    while b:
        a, b = b, a % b
    return a

def pairwise_coprime(mods: list[int]) -> bool:
    """Return True if every pair in mods is coprime."""
    for i in range(len(mods)):
        for j in range(i + 1, len(mods)):
            if gcd(mods[i], mods[j]) != 1:
                return False
    return True

def select_moduli(min_M: int,
                  candidates: list[int] | None = None) -> list[int]:
    if candidates is None:
        # Pairwise-coprime friendly candidates (powers-of-2 ± 1, primes)
        candidates = [127, 128, 255, 257, 511, 1021, 2039, 4093, 8191,
                      16381, 32749, 65521]

    # Filter to only prime/odd candidates – 128 = 2^7 coprime to odd moduli
    selected: list[int] = []
    product = 1
    for m in sorted(candidates, reverse=True):   # largest first → fewer moduli
        # Check coprimality with already selected
        if all(gcd(m, s) == 1 for s in selected):
            selected.append(m)
            product *= m
            if product > min_M:
                return sorted(selected)

    raise ValueError(
        f"Cannot reach dynamic range {min_M} with available candidates. "
        "Add larger primes to the candidate list."
    )

#### - CRT reconstruction

In [4]:
def crt_reconstruct(residues: list[np.ndarray], moduli: list[int]) -> np.ndarray:

    n = len(moduli)

    mixed = [r.astype(object) for r in residues]   # object dtype → arbitrary int

    for i in range(1, n):

        partial = mixed[0] % moduli[i]
        prod = 1
        for j in range(1, i):
            prod = (prod * moduli[j - 1]) % moduli[i]
            partial = (partial + prod * (mixed[j] % moduli[i])) % moduli[i]

        # correction: (residues[i] - partial) * inv(prod*m_{i-1}, m_i)  mod m_i
        prod = (prod * moduli[i - 1]) % moduli[i]
        inv  = pow(int(prod), -1, moduli[i])     # modular inverse
        mixed[i] = ((residues[i].astype(object) - partial) * inv) % moduli[i]

    # Now reconstruct from mixed-radix: x = a0 + m0*(a1 + m1*(a2+...))
    result = mixed[n - 1].copy()
    for i in range(n - 2, -1, -1):
        result = mixed[i] + moduli[i] * result

    return result.astype(object)   # keep as Python big-int array for safety


def signed_correction(x_unsigned, M: int) -> np.ndarray:
    """
    Convert unsigned CRT output in [0, M) to signed integer.
    Values > M//2 wrap around to negative.
    """
    half = M // 2
    # vectorised using numpy; x_unsigned may be object dtype
    result = np.where(x_unsigned > half,
                      x_unsigned - M,
                      x_unsigned)
    return result.astype(np.int64)

#### - RNS convolution (single modulus)

In [5]:
def rns_conv2d_single_modulus(
    x_mod: np.ndarray,
    w_mod: np.ndarray,
    b_mod: np.ndarray,
    m: int,
    padding: int = 1
) -> np.ndarray:

    # Convert to float64 tensors for PyTorch conv (keeps exact integers < 2^53)
    x_t = torch.from_numpy(x_mod.astype(np.float64)).unsqueeze(0)  # (1,C,H,W)
    w_t = torch.from_numpy(w_mod.astype(np.float64))
    b_t = torch.from_numpy(b_mod.astype(np.float64))

    with torch.no_grad():
        y_t = F.conv2d(x_t, w_t, bias=b_t, padding=padding)  # (1,C_out,H,W)

    y_np = y_t.squeeze(0).numpy()   # (C_out, H, W)

    # Reduce mod m and ensure non-negative residue
    y_mod = np.mod(np.round(y_np).astype(np.int64), m)
    return y_mod.astype(np.int64)

#### - Integer convolution reference

In [6]:
def integer_conv2d_reference(
    x_int: np.ndarray,   # (C_in, H, W)
    w_int: np.ndarray,   # (C_out, C_in, kH, kW)
    b_int: np.ndarray,   # (C_out,)
    padding: int = 1
) -> np.ndarray:

    x_t = torch.from_numpy(x_int.astype(np.float64)).unsqueeze(0)
    w_t = torch.from_numpy(w_int.astype(np.float64))
    b_t = torch.from_numpy(b_int.astype(np.float64))

    with torch.no_grad():
        y_t = F.conv2d(x_t, w_t, bias=b_t, padding=padding)

    return np.round(y_t.squeeze(0).numpy()).astype(np.int64)


#### - Pretty-print table

In [7]:
def print_table(rows: list[tuple]):
    col_w = max(len(str(r[0])) for r in rows) + 2
    val_w = max(len(str(r[1])) for r in rows) + 2
    sep = "+" + "-" * col_w + "+" + "-" * val_w + "+"
    print(sep)
    for label, value in rows:
        print(f"| {str(label):<{col_w-1}}| {str(value):>{val_w-1}}|")
    print(sep)

#### - RNS validation

In [8]:
def validate_rns_conv_layer(
    layer_name: str,
    x_fp32: np.ndarray,
    weight_file: str,
    bias_file: str,
    input_scale: int,
    padding: int,
    debug_dir: str,
) -> dict:

    tag = layer_name   # short prefix for printing
    print(f"\n{'='*60}")
    print(f"  {tag.upper()} RNS VALIDATION")
    print(f"{'='*60}")

    # ── 1. Quantise input ────────────────────────────────────────────────────
    # Only the input is quantised here; weights/bias are loaded pre-quantised.
    x_int = np.round(x_fp32 * input_scale).astype(np.int64)
    print(f"  Input shape  : {x_fp32.shape}  →  x_int {x_int.shape}")
    print(f"  x_int range  : [{x_int.min()}, {x_int.max()}]")

    # ── 2. Load pre-quantised weights & bias from Step 2 ────────────────────
    for path in (weight_file, bias_file):
        if not os.path.isfile(path):
            raise FileNotFoundError(
                f"[{tag}] Missing quantized parameter file: '{path}'. "
                "Run the Step 2 quantization script first."
            )

    # No further scaling, rounding, ceil, or floor applied.
    w_int = np.load(weight_file).astype(np.int64)
    b_int = np.load(bias_file).astype(np.int64)
    print(f"  Weight shape : {w_int.shape}  range [{w_int.min()}, {w_int.max()}]")
    print(f"  Bias   shape : {b_int.shape}  range [{b_int.min()}, {b_int.max()}]")

    # ── 3. Integer convolution reference ────────────────────────────────────
    # This is the ground truth the RNS result must match exactly.
    y_int_ref = integer_conv2d_reference(x_int, w_int, b_int, padding=padding)
    max_abs   = int(np.max(np.abs(y_int_ref)))
    print(f"  y_int_ref shape    : {y_int_ref.shape}")
    print(f"  max|y_int_ref|     : {max_abs:,}")

    # ── 4. Select moduli ─────────────────────────────────────────────────────
    # Require M > 2 * max_abs so signed CRT reconstruction is injective.
    required_M = 2 * max_abs + 1
    moduli     = select_moduli(required_M)
    M          = math.prod(moduli)
    assert M > 2 * max_abs, (
        f"[{tag}] Dynamic range insufficient! M={M} <= 2*max_abs={2*max_abs}."
    )
    assert pairwise_coprime(moduli), f"[{tag}] Moduli are not pairwise coprime!"
    print(f"  Moduli             : {moduli}")
    print(f"  Dynamic range M    : {M:,}")

    # ── 5. RNS encoding ──────────────────────────────────────────────────────
    # Python's % always returns a non-negative residue, even for negative ints.
    x_rns = [x_int % m for m in moduli]
    w_rns = [w_int % m for m in moduli]
    b_rns = [b_int % m for m in moduli]

    # ── 6. Per-modulus convolution ───────────────────────────────────────────
    # y_i = (x mod m_i  ⊛  w mod m_i  +  b mod m_i) mod m_i
    # Each channel is independent — the core RNS parallelism property.
    print(f"  Computing RNS convolutions …")
    y_rns_channels: list[np.ndarray] = []
    for idx, m in enumerate(moduli):
        y_i = rns_conv2d_single_modulus(
            x_rns[idx], w_rns[idx], b_rns[idx], m, padding=padding
        )
        y_rns_channels.append(y_i)
        print(f"    m={m:>6}  residue range [{y_i.min()}, {y_i.max()}]")

    # ── 7. CRT reconstruction ────────────────────────────────────────────────
    # Garner's algorithm recovers unique x in [0, M) from its residues.
    y_unsigned = crt_reconstruct(y_rns_channels, moduli)

    # ── 8. Signed correction ─────────────────────────────────────────────────
    # Map values > M//2 back to negative: covers the full signed range.
    y_rns_reconstructed = signed_correction(y_unsigned, M)

    # ── 9. Metrics ───────────────────────────────────────────────────────────
    diff       = y_rns_reconstructed.astype(np.int64) - y_int_ref.astype(np.int64)
    n_total    = diff.size
    n_mismatch = int(np.count_nonzero(diff))
    exact_pct  = 100.0 * (n_total - n_mismatch) / n_total
    mae        = float(np.mean(np.abs(diff)))
    mse        = float(np.mean(diff.astype(np.float64) ** 2))
    max_err    = int(np.max(np.abs(diff)))

    rows = [
        ("Layer",                 tag),
        ("Input shape",           str(x_fp32.shape)),
        ("Weight shape",          str(w_int.shape)),
        ("Output shape",          str(y_int_ref.shape)),
        ("Moduli",                moduli),
        ("Dynamic range M",       f"{M:,}"),
        ("Max abs accumulator",   f"{max_abs:,}"),
        ("M > 2×max_abs",         M > 2 * max_abs),
        ("Total elements",        n_total),
        ("Mismatched elements",   n_mismatch),
        ("Exact match %",         f"{exact_pct:.4f} %"),
        ("MAE",                   f"{mae:.6f}"),
        ("MSE",                   f"{mse:.6f}"),
        ("Max abs error",         max_err),
    ]
    print()
    print_table(rows)

    if n_mismatch > 0:
        print(f"\n⚠  [{tag}] Mismatch! Check moduli range or implementation.")
    else:
        print(f"\n✓  [{tag}] Perfect match: RNS output == integer reference.")

    # ── 10. Save debug artefacts ─────────────────────────────────────────────
    os.makedirs(debug_dir, exist_ok=True)
    np.save(f"{debug_dir}/{tag}_x_int_sample.npy",       x_int)
    np.save(f"{debug_dir}/{tag}_w_int.npy",               w_int)
    np.save(f"{debug_dir}/{tag}_b_int.npy",               b_int)
    np.save(f"{debug_dir}/{tag}_y_int_reference.npy",     y_int_ref)
    np.save(f"{debug_dir}/{tag}_y_rns_reconstructed.npy", y_rns_reconstructed)

    # Return metrics dict for the combined metadata JSON
    return {
        "input_shape":         str(x_fp32.shape),
        "weight_shape":        str(w_int.shape),
        "output_shape":        str(y_int_ref.shape),
        "weight_file":         weight_file,
        "bias_file":           bias_file,
        "moduli":              moduli,
        "dynamic_range_M":     str(M),
        "max_abs_accumulator": max_abs,
        "exact_match_pct":     exact_pct,
        "n_mismatch":          n_mismatch,
        "mae":                 mae,
        "max_abs_error":       max_err,
    }

In [9]:
def rns_conv2d_reconstruct(
    x_int: np.ndarray,    # (C_in, H, W)   int64
    w_int: np.ndarray,    # (C_out, C_in, kH, kW)  int64
    b_int: np.ndarray,    # (C_out,)  int64
    padding: int = 1,
    debug: bool = False,
) -> np.ndarray:
    # ── Optional debug cross-check ───────────────────────────────────────────
    if debug:
        y_ref = integer_conv2d_reference(x_int, w_int, b_int, padding=padding)
        max_abs_ref = int(np.max(np.abs(y_ref)))
    else:
        # Still need max_abs to size the moduli, but compute from RNS output
        # after reconstruction (chicken-and-egg: we run a fast FP reference).
        y_ref = integer_conv2d_reference(x_int, w_int, b_int, padding=padding)
        max_abs_ref = int(np.max(np.abs(y_ref)))
        # Note: this reference is computed purely to choose moduli; we then
        # discard it and use the RNS reconstruction as the actual output.

    # ── Moduli selection ─────────────────────────────────────────────────────
    # M > 2 * max_abs guarantees injective signed CRT reconstruction.
    required_M = 2 * max_abs_ref + 1
    moduli     = select_moduli(required_M)
    M          = math.prod(moduli)

    # ── RNS encode ───────────────────────────────────────────────────────────
    # Python % returns non-negative residue for any sign of the input.
    x_rns = [x_int % m for m in moduli]
    w_rns = [w_int % m for m in moduli]
    b_rns = [b_int % m for m in moduli]

    # ── Per-modulus convolution ───────────────────────────────────────────────
    # y_i = (x mod m_i  ⊛  w mod m_i  +  b mod m_i) mod m_i
    y_channels = [
        rns_conv2d_single_modulus(x_rns[i], w_rns[i], b_rns[i], moduli[i],
                                  padding=padding)
        for i in range(len(moduli))
    ]

    # ── CRT + signed correction ───────────────────────────────────────────────
    y_unsigned     = crt_reconstruct(y_channels, moduli)
    y_reconstructed = signed_correction(y_unsigned, M)   # int64

    # ── Optional debug validation ─────────────────────────────────────────────
    if debug:
        diff      = y_reconstructed.astype(np.int64) - y_ref.astype(np.int64)
        n_mis     = int(np.count_nonzero(diff))
        exact_pct = 100.0 * (diff.size - n_mis) / diff.size
        print(f"    [rns_conv2d_reconstruct DEBUG] "
              f"exact={exact_pct:.2f}%  max_err={int(np.max(np.abs(diff)))}")

    return y_reconstructed.astype(np.int64)


def inference_rns_assisted(
    model,
    x_fp32_t: torch.Tensor,
    w1_int: np.ndarray,
    b1_int: np.ndarray,
    w2_int: np.ndarray,
    b2_int: np.ndarray,
    input_scale: int,
    weight_scale: int,
    padding: int = 1,
    debug: bool = False,
) -> tuple[torch.Tensor, dict]:

    combined_scale = float(input_scale * weight_scale)
    internals: dict = {}

    # ── Conv1 ─────────────────────────────────────────────────────────────────
    x1_fp32 = x_fp32_t.squeeze(0).numpy()               # (1, 28, 28)
    x1_int  = np.round(x1_fp32 * input_scale).astype(np.int64)

    y1_int  = rns_conv2d_reconstruct(x1_int, w1_int, b1_int,
                                      padding=padding, debug=debug)

    # Rescale integer accumulator back to FP32-compatible magnitude.
    # y_int = x_int * w_int ≈ (x_fp32 * Is) * (w_fp32 * Ws)
    #       = x_fp32 * w_fp32 * Is * Ws
    # → y_fp32_approx = y_int / (Is * Ws)
    y1_fp32 = y1_int.astype(np.float32) / combined_scale  # (32, 28, 28)
    internals["conv1_pre_relu_rns"] = y1_fp32

    # ReLU + MaxPool in PyTorch (FP32)
    y1_t = torch.from_numpy(y1_fp32).unsqueeze(0)         # (1, 32, 28, 28)
    a1   = F.relu(y1_t)
    p1   = F.max_pool2d(a1, kernel_size=2, stride=2)      # (1, 32, 14, 14)
    internals["conv1_post_pool_rns"] = p1.squeeze(0).numpy()

    # ── Conv2 ─────────────────────────────────────────────────────────────────
    # Conv2 input = post-pool activations from Conv1, re-quantised to int.
    x2_fp32 = p1.squeeze(0).numpy()                       # (32, 14, 14)
    x2_int  = np.round(x2_fp32 * input_scale).astype(np.int64)

    y2_int  = rns_conv2d_reconstruct(x2_int, w2_int, b2_int,
                                      padding=padding, debug=debug)

    y2_fp32 = y2_int.astype(np.float32) / combined_scale  # (64, 14, 14)
    internals["conv2_pre_relu_rns"] = y2_fp32

    # ReLU + MaxPool in PyTorch (FP32)
    y2_t = torch.from_numpy(y2_fp32).unsqueeze(0)         # (1, 64, 14, 14)
    a2   = F.relu(y2_t)
    p2   = F.max_pool2d(a2, kernel_size=2, stride=2)      # (1, 64, 7, 7)

    # ── FP32 classifier ───────────────────────────────────────────────────────
    # Dropout is deterministic in eval mode (identity), so no special handling.
    model.eval()
    with torch.no_grad():
        logits = model.classifier(p2)                      # (1, 10)

    return logits, internals


def evaluate_rns_assisted(
    model,
    test_loader: torch.utils.data.DataLoader,
    w1_int: np.ndarray,
    b1_int: np.ndarray,
    w2_int: np.ndarray,
    b2_int: np.ndarray,
    input_scale: int,
    weight_scale: int,
    padding: int = 1,
    max_samples: int | None = None,
) -> dict:

    rns_correct  = 0
    fp32_correct = 0
    total        = 0

    model.eval()
    for images, labels in test_loader:
        for img_t, lbl in zip(images, labels):
            if max_samples is not None and total >= max_samples:
                break

            # FP32 baseline
            with torch.no_grad():
                logits_fp32 = model(img_t.unsqueeze(0))
            pred_fp32 = int(logits_fp32.argmax(dim=1).item())
            fp32_correct += int(pred_fp32 == lbl.item())

            # RNS-assisted
            logits_rns, _ = inference_rns_assisted(
                model, img_t,
                w1_int, b1_int, w2_int, b2_int,
                input_scale, weight_scale, padding,
                debug=False,
            )
            pred_rns = int(logits_rns.argmax(dim=1).item())
            rns_correct += int(pred_rns == lbl.item())

            total += 1

        if max_samples is not None and total >= max_samples:
            break

    rns_acc  = 100.0 * rns_correct  / total
    fp32_acc = 100.0 * fp32_correct / total
    return {
        "total":        total,
        "rns_correct":  rns_correct,
        "fp32_correct": fp32_correct,
        "rns_acc":      rns_acc,
        "fp32_acc":     fp32_acc,
        "acc_loss":     fp32_acc - rns_acc,
    }

#### - visualisation

In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import json

def generate_rns_visual_report(metadata_path="rns_debug/rns_metadata.json",
                               out_dir="rns_debug/figures"):
    os.makedirs(out_dir, exist_ok=True)

    with open(metadata_path, "r") as f:
        metadata = json.load(f)

    conv1 = metadata["conv1"]
    conv2 = metadata["conv2"]
    single = metadata["step4_single_image"]
    acc = metadata["step4_accuracy"]

    # =========================
    # Table 1 — Layer Validation
    # =========================
    layer_df = pd.DataFrame([
        {
            "Layer": "Conv1",
            "Input shape": str(conv1["input_shape"]),
            "Output shape": str(conv1["output_shape"]),
            "Dynamic range M": conv1["dynamic_range_M"],
            "Max abs accumulator": conv1["max_abs_accumulator"],
            "M > 2×max_abs": conv1["dynamic_range_M"] > 2 * conv1["max_abs_accumulator"],
            "Exact match (%)": conv1["exact_match_pct"],
            "MAE": conv1["mae"],
            "Max abs error": conv1["max_abs_error"],
        },
        {
            "Layer": "Conv2",
            "Input shape": str(conv2["input_shape"]),
            "Output shape": str(conv2["output_shape"]),
            "Dynamic range M": conv2["dynamic_range_M"],
            "Max abs accumulator": conv2["max_abs_accumulator"],
            "M > 2×max_abs": conv2["dynamic_range_M"] > 2 * conv2["max_abs_accumulator"],
            "Exact match (%)": conv2["exact_match_pct"],
            "MAE": conv2["mae"],
            "Max abs error": conv2["max_abs_error"],
        }
    ])

    layer_df.to_csv(f"{out_dir}/layer_validation_table.csv", index=False)

    # Save table as image
    fig, ax = plt.subplots(figsize=(12, 2.8))
    ax.axis("off")
    table = ax.table(
        cellText=layer_df.values,
        colLabels=layer_df.columns,
        loc="center",
        cellLoc="center"
    )
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.5)
    plt.tight_layout()
    plt.savefig(f"{out_dir}/layer_validation_table.png", dpi=300)
    plt.close()

    # =========================
    # Graph 1 — Exact Match %
    # =========================
    plt.figure(figsize=(6, 4))
    plt.bar(["Conv1", "Conv2"], [
        conv1["exact_match_pct"],
        conv2["exact_match_pct"]
    ])
    plt.ylabel("Exact Match (%)")
    plt.ylim(0, 105)
    plt.title("RNS Reconstruction Exact Match per Convolution Layer")
    plt.tight_layout()
    plt.savefig(f"{out_dir}/exact_match_bar.png", dpi=300)
    plt.close()

    # =========================
    # Graph 2 — Dynamic Range Safety
    # =========================
    plt.figure(figsize=(7, 4))
    layers = ["Conv1", "Conv2"]
    max_abs = [
        conv1["max_abs_accumulator"],
        conv2["max_abs_accumulator"]
    ]
    half_M = [
        conv1["dynamic_range_M"] / 2,
        conv2["dynamic_range_M"] / 2
    ]

    x = range(len(layers))
    plt.bar([i - 0.2 for i in x], max_abs, width=0.4, label="Max |Accumulator|")
    plt.bar([i + 0.2 for i in x], half_M, width=0.4, label="M / 2")
    plt.xticks(x, layers)
    plt.ylabel("Value")
    plt.title("Dynamic Range Safety Check")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{out_dir}/dynamic_range_safety.png", dpi=300)
    plt.close()

    # =========================
    # Graph 3 — Accuracy Comparison
    # =========================
    plt.figure(figsize=(6, 4))
    plt.bar(["FP32 Baseline", "RNS-assisted"], [
        acc["fp32_acc"],
        acc["rns_acc"]
    ])
    plt.ylabel("Accuracy (%)")
    plt.ylim(98.5, 100)
    plt.title("FP32 vs RNS-assisted Test Accuracy")
    plt.tight_layout()
    plt.savefig(f"{out_dir}/accuracy_comparison.png", dpi=300)
    plt.close()

    # =========================
    # Graph 4 — Error Metrics
    # =========================
    error_df = pd.DataFrame({
        "Metric": [
            "Logit MAE",
            "Logit max abs error",
            "Conv1 pre-ReLU MAE",
            "Conv2 pre-ReLU MAE"
        ],
        "Value": [
            single["logit_mae"],
            single["logit_max"],
            single["conv1_pre_relu_mae"],
            single["conv2_pre_relu_mae"]
        ]
    })

    error_df.to_csv(f"{out_dir}/single_image_error_metrics.csv", index=False)

    plt.figure(figsize=(8, 4))
    plt.bar(error_df["Metric"], error_df["Value"])
    plt.ylabel("Error")
    plt.title("Single-image FP32 vs RNS-assisted Error Metrics")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.savefig(f"{out_dir}/single_image_error_metrics.png", dpi=300)
    plt.close()

    print(f"\nVisual report saved in: {out_dir}")

#### - Main

In [19]:
def main():
    # ── Config ───────────────────────────────────────────────────────────────
    INPUT_SCALE   = 2 ** 8      # scale factor applied to normalised input pixels
    PADDING       = 1

    BASELINE_PTH  = "cnn_bns_hw_baseline.pth"
    BASELINE_META = "cnn_bns_hw_baseline_metadata.json"
    QUANT_DIR     = "quantized_params"
    DEBUG_DIR     = "rns_debug"

    # ── Load CNN_BNS_HW ──────────────────────────────────────────────────────
    print("=" * 60)
    print("Loading trained CNN_BNS_HW …")
    if not os.path.isfile(BASELINE_PTH):
        raise FileNotFoundError(
            f"Could not find '{BASELINE_PTH}'. "
            "Place the baseline checkpoint in the working directory."
        )

    model = CNN_BNS_HW(
        in_channels=1,
        num_classes=10,
        conv1_cfg=(32, 3, 1, 1),
        conv2_cfg=(64, 3, 1, 1),
        fc1_units=256,
        dropout_rate=0.5,
        input_hw=(28, 28),
    )
    state = torch.load(BASELINE_PTH, map_location="cpu")
    model.load_state_dict(state)
    model.eval()
    print(f"  Loaded '{BASELINE_PTH}' ✓")

    if os.path.isfile(BASELINE_META):
        with open(BASELINE_META) as f:
            print(f"  Baseline metadata: {json.load(f)}")

    # ── Load MNIST test image (same normalisation as training) ───────────────
    print("\nLoading MNIST test image …")
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])
    test_set   = datasets.MNIST(root="./data", train=False,
                                download=True, transform=transform)
    x_fp32_t, label = test_set[0]          # index 0; change as needed
    x_fp32          = x_fp32_t.numpy()     # (1, 28, 28)
    print(f"  Label       : {label}")
    print(f"  Pixel range : [{x_fp32.min():.4f}, {x_fp32.max():.4f}]")

    # ── Run model to capture all feature maps ────────────────────────────────
    # Required for:
    #   • sanity-checking Conv1 FP32 output  (informational)
    #   • supplying conv1.post_pool as the quantised Conv2 input
    print("\nRunning model forward pass to capture feature maps …")
    with torch.no_grad():
        _, fmaps = model(x_fp32_t.unsqueeze(0), return_features=True)

    conv1_fp32_out = fmaps["conv1.pre_relu"].squeeze(0).numpy()   # (32,28,28)
    print(f"  Conv1 FP32 pre-relu range : "
          f"[{conv1_fp32_out.min():.4f}, {conv1_fp32_out.max():.4f}]")

    # Conv2 input: FP32 post-pool output of Conv Block 1 (after ReLU + MaxPool)
    # Shape: (32, 14, 14)  — this is what feeds into Conv2 in the real model.
    x_conv2_fp32 = fmaps["conv1.post_pool"].squeeze(0).numpy()    # (32,14,14)
    print(f"  Conv1 post-pool range     : "
          f"[{x_conv2_fp32.min():.4f}, {x_conv2_fp32.max():.4f}]")

    # ── Conv1 RNS validation ─────────────────────────────────────────────────
    meta_conv1 = validate_rns_conv_layer(
        layer_name  = "conv1",
        x_fp32      = x_fp32,                                            # (1,28,28)
        weight_file = f"{QUANT_DIR}/conv1_weight_int.npy",
        bias_file   = f"{QUANT_DIR}/conv1_bias_int.npy",
        input_scale = INPUT_SCALE,
        padding     = PADDING,
        debug_dir   = DEBUG_DIR,
    )

    # ── Conv2 RNS validation ─────────────────────────────────────────────────
    # Input is the FP32 post-pool feature map from Conv Block 1.
    # It will be quantised inside validate_rns_conv_layer using the same
    # input_scale, matching the integer pipeline expectation.
    meta_conv2 = validate_rns_conv_layer(
        layer_name  = "conv2",
        x_fp32      = x_conv2_fp32,                                      # (32,14,14)
        weight_file = f"{QUANT_DIR}/conv2_weight_int.npy",
        bias_file   = f"{QUANT_DIR}/conv2_bias_int.npy",
        input_scale = INPUT_SCALE,
        padding     = PADDING,
        debug_dir   = DEBUG_DIR,
    )

    # ── Summary ──────────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print("  SUMMARY")
    print(f"{'='*60}")
    for layer, meta in [("Conv1", meta_conv1), ("Conv2", meta_conv2)]:
        status = "✓ PASS" if meta["n_mismatch"] == 0 else "⚠ FAIL"
        print(f"  {layer}: {status}  |  "
              f"exact={meta['exact_match_pct']:.4f}%  "
              f"MAE={meta['mae']:.6f}  "
              f"max_err={meta['max_abs_error']}")

    # ── Combined metadata JSON ───────────────────────────────────────────────
    os.makedirs(DEBUG_DIR, exist_ok=True)
    metadata = {
        "baseline_checkpoint": BASELINE_PTH,
        "input_scale":         INPUT_SCALE,
        "label":               int(label),
        "conv1":               meta_conv1,
        "conv2":               meta_conv2,
    }
    meta_path = f"{DEBUG_DIR}/rns_metadata.json"
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)
    print(f"\n  Metadata saved → {meta_path}")

    # ═════════════════════════════════════════════════════════════════════════
    # STEP 4 — RNS-ASSISTED INFERENCE
    # ═════════════════════════════════════════════════════════════════════════

    # ── Load weight scale (from quantization_metadata.json if available) ─────
    quant_meta_path = f"{QUANT_DIR}/quantization_metadata.json"
    if os.path.isfile(quant_meta_path):
        with open(quant_meta_path) as f:
            qmeta = json.load(f)
        WEIGHT_SCALE = int(qmeta.get("weight_scale", 2 ** 11))
        print(f"\n  Weight scale loaded from metadata: {WEIGHT_SCALE}")
    else:
        WEIGHT_SCALE = 2 ** 11
        print(f"\n  Weight scale defaulting to 2^11 = {WEIGHT_SCALE}")

    # ── Load pre-quantised weight arrays (already loaded for validation,
    #    but we reload cleanly here so Step 4 is self-contained) ─────────────
    w1_int = np.load(f"{QUANT_DIR}/conv1_weight_int.npy").astype(np.int64)
    b1_int = np.load(f"{QUANT_DIR}/conv1_bias_int.npy").astype(np.int64)
    w2_int = np.load(f"{QUANT_DIR}/conv2_weight_int.npy").astype(np.int64)
    b2_int = np.load(f"{QUANT_DIR}/conv2_bias_int.npy").astype(np.int64)

    # ── 4a. Single-image comparison ──────────────────────────────────────────
    print(f"\n{'='*60}")
    print("  STEP 4a — Single-image RNS-assisted inference")
    print(f"{'='*60}")

    # FP32 baseline for this image
    with torch.no_grad():
        logits_fp32, fmaps_fp32 = model(x_fp32_t.unsqueeze(0),
                                        return_features=True)
    pred_fp32   = int(logits_fp32.argmax(dim=1).item())
    conf_fp32   = float(torch.softmax(logits_fp32, dim=1).max().item())
    c1_pre_fp32 = fmaps_fp32["conv1.pre_relu"].squeeze(0).numpy()
    c2_pre_fp32 = fmaps_fp32["conv2.pre_relu"].squeeze(0).numpy()

    # RNS-assisted inference (debug=True prints per-layer exact-match stats)
    logits_rns, internals = inference_rns_assisted(
        model, x_fp32_t,
        w1_int, b1_int, w2_int, b2_int,
        input_scale=INPUT_SCALE, weight_scale=WEIGHT_SCALE,
        padding=PADDING, debug=True,
    )
    pred_rns  = int(logits_rns.argmax(dim=1).item())
    conf_rns  = float(torch.softmax(logits_rns, dim=1).max().item())

    # Per-layer pre-relu MAE (FP32 vs RNS-rescaled)
    c1_pre_rns = internals["conv1_pre_relu_rns"]
    c2_pre_rns = internals["conv2_pre_relu_rns"]
    c1_mae     = float(np.mean(np.abs(c1_pre_rns - c1_pre_fp32)))
    c2_mae     = float(np.mean(np.abs(c2_pre_rns - c2_pre_fp32)))

    logit_diff = (logits_rns.numpy() - logits_fp32.numpy()).squeeze()
    logit_mae  = float(np.mean(np.abs(logit_diff)))
    logit_max  = float(np.max(np.abs(logit_diff)))

    rows_single = [
        ("Ground-truth label",         int(label)),
        ("FP32 predicted class",        pred_fp32),
        ("RNS-assisted predicted class",pred_rns),
        ("FP32 confidence",             f"{conf_fp32:.4f}"),
        ("RNS-assisted confidence",     f"{conf_rns:.4f}"),
        ("Logit MAE",                   f"{logit_mae:.6f}"),
        ("Logit max abs error",         f"{logit_max:.6f}"),
        ("Conv1 pre-relu MAE",          f"{c1_mae:.6f}"),
        ("Conv2 pre-relu MAE",          f"{c2_mae:.6f}"),
    ]
    print()
    print_table(rows_single)

    # ── 4b. Test-set accuracy evaluation ────────────────────────────────────
    print(f"\n{'='*60}")
    print("  STEP 4b — Test-set accuracy (full test set)")
    print(f"{'='*60}")

    MAX_SAMPLES = None   # increase to 1000 or None for fuller evaluation

    # Reload test_set with a DataLoader (batch_size=1 keeps the loop simple)
    test_loader = torch.utils.data.DataLoader(
        test_set, batch_size=32, shuffle=False
    )

    print(f"\n  Running RNS-assisted inference on {MAX_SAMPLES} test images …")
    acc_results = evaluate_rns_assisted(
        model, test_loader,
        w1_int, b1_int, w2_int, b2_int,
        input_scale=INPUT_SCALE,
        weight_scale=WEIGHT_SCALE,
        padding=PADDING,
        max_samples=MAX_SAMPLES,
    )

    rows_acc = [
        ("Mode",          "Samples", "Correct", "Accuracy (%)"),
    ]
    print()
    # Header
    hdr = f"{'Mode':<22} {'Samples':>8} {'Correct':>8} {'Accuracy':>12}"
    sep = "-" * len(hdr)
    print(hdr)
    print(sep)
    n = acc_results["total"]
    print(f"{'FP32 baseline':<22} {n:>8} {acc_results['fp32_correct']:>8} "
          f"{acc_results['fp32_acc']:>11.2f}%")
    print(f"{'RNS-assisted':<22} {n:>8} {acc_results['rns_correct']:>8} "
          f"{acc_results['rns_acc']:>11.2f}%")
    print(sep)
    print(f"{'Accuracy loss':<22} {'':>8} {'':>8} "
          f"{acc_results['acc_loss']:>+11.2f}%")

    # ── Persist Step 4 results into metadata JSON ────────────────────────────
    metadata["step4_single_image"] = {
        "fp32_pred":    pred_fp32,
        "rns_pred":     pred_rns,
        "fp32_conf":    conf_fp32,
        "rns_conf":     conf_rns,
        "logit_mae":    logit_mae,
        "logit_max":    logit_max,
        "conv1_pre_relu_mae": c1_mae,
        "conv2_pre_relu_mae": c2_mae,
    }
    metadata["step4_accuracy"] = acc_results
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)
    print(f"\n  Updated metadata saved → {meta_path}")


    print("\nDone.")


# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    main()


Loading trained CNN_BNS_HW …
  Loaded 'cnn_bns_hw_baseline.pth' ✓
  Baseline metadata: {'run_id': '20260429_050342', 'model_variant': 'dual', 'config': {'seed': '42', 'dataset': 'MNIST', 'data_dir': './data', 'num_workers': '2', 'batch_size': '64', 'learning_rate': '0.001', 'num_epochs': '10', 'weight_decay': '0.0001', 'conv1': '(32, 3, 1, 1)', 'conv2': '(64, 3, 1, 1)', 'fc1_units': '256', 'dropout_rate': '0.5', 'model_variant': 'dual', 'model_save_path': './cnn_bns_hw_baseline.pth', 'metadata_save_path': './cnn_bns_hw_baseline_metadata.json', 'device': 'cpu'}, 'architecture': {'type': 'CNN_BNS_HW (dual-conv)', 'batch_norm': False, 'conv_bias': True, 'conv_block1': '1→32 ch, k=3, pad=1', 'conv_block2': '32→64 ch, k=3, pad=1', 'flatten_dim': 3136, 'fc1': '3136→256', 'fc2_out': '256→10', 'total_params': 824458}, 'total_params': 824458, 'training_history': [{'epoch': 1, 'train_loss': 0.166468, 'train_acc': 94.8033, 'test_acc': 98.76, 'lr': 0.001}, {'epoch': 2, 'train_loss': 0.05986, 'trai